# Agentic Mobility - Visualization Demo

This notebook demonstrates how to visualize the mobility data generated by the Agentic Mobility system using the legacy visualization functions.

## Setup

First, we'll set up the paths and import necessary libraries.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Add the maveric root directory to the path
# This assumes the notebook is in radp/digital_twin/agentic_mobility/examples/
maveric_root = Path.cwd().parent.parent.parent.parent
if str(maveric_root) not in sys.path:
    sys.path.insert(0, str(maveric_root))

print(f"Maveric root: {maveric_root}")
print(f"Current working directory: {Path.cwd()}")

In [ ]:
# Import the legacy visualization functions
from radp.digital_twin.agentic_mobility.visualization.legacy import (
    plot_ue_tracks,
    plot_ue_tracks_side_by_side,
    plot_ue_tracks_on_axis
)

print("Successfully imported visualization functions!")

## Load Generated Data

Let's load the CSV and metadata files from the `generated_ues` directory.

> if no available csv file in `data_dir` please run the `end_to_end_example.py`

In [ ]:
# Define the data directory
data_dir = Path.cwd() / "generated_ues"

# List all available CSV files
csv_files = list(data_dir.glob("*.csv"))

print(f"Data directory: {data_dir}")
print(f"\nAvailable CSV files ({len(csv_files)}):")
for i, csv_file in enumerate(csv_files, 1):
    print(f"  {i}. {csv_file.name}")

In [ ]:
# Select the first CSV file (or modify the index to select a different one)
if csv_files:
    selected_csv = csv_files[0]
    print(f"Loading: {selected_csv.name}")
    
    # Load the CSV
    df = pd.read_csv(selected_csv)
    
    # Try to load corresponding metadata
    metadata_file = selected_csv.with_name(selected_csv.stem + "_metadata.json")
    if metadata_file.exists():
        with open(metadata_file, 'r') as f:
            metadata = json.load(f)
        print(f"Loaded metadata: {metadata_file.name}")
    else:
        metadata = None
        print("No metadata file found")
else:
    print("No CSV files found in the generated_ues directory.")
    print("Please run the end_to_end_example.py script first to generate data.")

## Explore the Data

Let's take a look at the structure and content of the loaded data.

In [ ]:
if csv_files:
    print("DataFrame Info:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"\nFirst 10 rows:")
    display(df.head(10))
    
    print(f"\nDataFrame Summary Statistics:")
    display(df.describe())

In [ ]:
if csv_files and metadata:
    print("Metadata:")
    print(json.dumps(metadata, indent=2))

## Visualization 1: Plot UE Tracks

This visualization shows the movement tracks of all UE IDs with directional arrows.
Each UE is shown in a different color, with arrows indicating the direction of movement.

**Function:** `plot_ue_tracks(df)`

**Features:**
- Displays UE movement over time with directional arrows
- Each UE gets a unique color from the color map
- Starting points are marked with circles
- If there are multiple batches (detected by tick resets), they are plotted separately

In [ ]:
if csv_files:
    # Enable inline plotting
    %matplotlib inline
    
    print("Plotting UE tracks...")
    plot_ue_tracks(df)

## Visualization 2: Side-by-Side Comparison

This visualization allows you to compare two different mobility datasets side by side.
This is useful for comparing different scenarios, parameters, or time periods.

**Function:** `plot_ue_tracks_side_by_side(df1, df2)`

**Features:**
- Plots two datasets side by side for easy comparison
- Same color coding for corresponding UE IDs (if applicable)
- Useful for A/B testing different mobility scenarios

In [ ]:
# Load a second CSV file if available for comparison
if len(csv_files) >= 2:
    selected_csv_2 = csv_files[1]
    print(f"Loading second dataset: {selected_csv_2.name}")
    df2 = pd.read_csv(selected_csv_2)
    
    print(f"\nDataset 1: {csv_files[0].name}")
    print(f"  - UEs: {df['mock_ue_id'].nunique()}")
    print(f"  - Ticks: {df['tick'].max() + 1}")
    print(f"  - Total points: {len(df)}")
    
    print(f"\nDataset 2: {csv_files[1].name}")
    print(f"  - UEs: {df2['mock_ue_id'].nunique()}")
    print(f"  - Ticks: {df2['tick'].max() + 1}")
    print(f"  - Total points: {len(df2)}")
else:
    print(f"Only {len(csv_files)} CSV file(s) available.")
    print("Need at least 2 CSV files for side-by-side comparison.")
    print("Run the end_to_end_example.py script multiple times with different queries to generate more datasets.")

In [ ]:
if len(csv_files) >= 2:
    print("Plotting side-by-side comparison...")
    plot_ue_tracks_side_by_side(df, df2)

## Custom Visualization with `plot_ue_tracks_on_axis`

This is a helper function that plots UE tracks on a given matplotlib axis.
You can use this to create custom multi-plot layouts.

**Function:** `plot_ue_tracks_on_axis(df, ax, title)`

**Features:**
- More flexible plotting on custom matplotlib axes
- Allows for creating complex multi-plot layouts
- Can be combined with other matplotlib plots

In [ ]:
if csv_files:
    # Create a custom layout with a single plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    plot_ue_tracks_on_axis(df, ax, title=f"Custom Plot: {selected_csv.name}")
    
    plt.tight_layout()
    plt.show()

## Advanced: Custom Multi-Plot Layout

Let's create a 2x2 grid showing different aspects of the data or multiple datasets.

In [ ]:
if len(csv_files) >= 4:
    # Create a 2x2 grid of plots
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    
    # Plot the first 4 datasets
    for i, (csv_file, ax) in enumerate(zip(csv_files[:4], axes.flat)):
        df_temp = pd.read_csv(csv_file)
        plot_ue_tracks_on_axis(df_temp, ax, title=csv_file.name)
    
    plt.tight_layout()
    plt.show()
elif len(csv_files) >= 2:
    # Create a 1x2 grid of plots
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    for i, (csv_file, ax) in enumerate(zip(csv_files[:2], axes.flat)):
        df_temp = pd.read_csv(csv_file)
        plot_ue_tracks_on_axis(df_temp, ax, title=csv_file.name)
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Only {len(csv_files)} dataset(s) available.")
    print("Generate more datasets to see multi-plot layouts.")

## Data Analysis: UE Movement Statistics

Let's analyze some statistics about the UE movements.

In [ ]:
if csv_files:
    # Calculate movement distances for each UE
    import numpy as np
    
    def calculate_distance(lat1, lon1, lat2, lon2):
        """Calculate Euclidean distance between two points (simplified)."""
        return np.sqrt((lat2 - lat1)**2 + (lon2 - lon1)**2)
    
    # Group by UE and calculate statistics
    ue_stats = []
    
    for ue_id in df['mock_ue_id'].unique():
        ue_data = df[df['mock_ue_id'] == ue_id].sort_values('tick')
        
        # Calculate total distance traveled
        distances = []
        for i in range(len(ue_data) - 1):
            lat1, lon1 = ue_data.iloc[i][['lat', 'lon']]
            lat2, lon2 = ue_data.iloc[i + 1][['lat', 'lon']]
            distances.append(calculate_distance(lat1, lon1, lat2, lon2))
        
        total_distance = sum(distances)
        avg_distance = np.mean(distances) if distances else 0
        
        ue_stats.append({
            'UE ID': ue_id,
            'Total Distance': total_distance,
            'Avg Distance per Tick': avg_distance,
            'Num Ticks': len(ue_data),
            'Start Lat': ue_data.iloc[0]['lat'],
            'Start Lon': ue_data.iloc[0]['lon'],
            'End Lat': ue_data.iloc[-1]['lat'],
            'End Lon': ue_data.iloc[-1]['lon']
        })
    
    stats_df = pd.DataFrame(ue_stats)
    print("\nUE Movement Statistics:")
    display(stats_df)

## Summary

This notebook demonstrated:

1. **Loading Data**: How to load generated mobility CSVs and metadata from the `generated_ues` directory
2. **Basic Visualization**: Using `plot_ue_tracks()` to visualize UE movement with directional arrows
3. **Comparison**: Using `plot_ue_tracks_side_by_side()` to compare two datasets
4. **Custom Layouts**: Using `plot_ue_tracks_on_axis()` to create custom multi-plot layouts
5. **Analysis**: Calculating and displaying UE movement statistics

### Next Steps

- Run `end_to_end_example.py` multiple times with different queries to generate more datasets
- Experiment with different visualization layouts
- Add your own custom analysis and visualizations
- Compare different scenarios (urban vs suburban vs highway, etc.)